In [6]:
import os
import json
from collections import defaultdict

import networkx as nx
from tqdm import tqdm
from nltk.corpus import wordnet as wn
import nltk
from matplotlib import pyplot as plt
import numpy as np
from node2vec import Node2Vec

### Node2Vec

#### Weighted Graph


##### Generate Random Walks

In [16]:
len_random_walk = 10
walks_per_node = 30
dim_embeddings = 200
p = 20
q = 0.25

In [17]:
def random_walk(start_node, graph, walk_length=len_random_walk):
    walk = []
    curr = start_node
    prev = start_node
    while len(walk) < walk_length:
        walk.append(curr)
        out_edges = [[edge[1], edge[2]['weight']] for edge in graph.out_edges(curr, data=True) if ((", 'meta'" not in edge[1]) \
                                                                                                    or (curr is start_node))
                    ]
        in_edges = [edge[0] for edge in graph.in_edges(curr) if edge[0] in walk]

        neighbors = [edge[0] for edge in out_edges] + in_edges

        if len(neighbors) == 0:
            return walk
            
        raw_probs = [edge[1] / q for edge in out_edges] + [1/p for edge in in_edges]
        probs = np.array(raw_probs) / sum(raw_probs)
        next = neighbors[np.random.choice(len(neighbors), p=probs)]
        prev = curr
        curr = next
        
    return walk


In [22]:
walks2 = []
num_nodes = len(graph.nodes())
loop = tqdm(total=num_nodes*walks_per_node)
k = 0
for node in graph.nodes():
    for i in range(walks_per_node):
        walk = []
        while len(walk) == 0:
            walk = random_walk(node, graph)
        walks.append(walk)

    k += 1
    if k % 5 == 0:
        loop.update(5*walks_per_node)
loop.close()
        

  0%|                                                                         | 750/7929690 [00:11<34:11:40, 64.41it/s]
node: ('human', 'n', 'singular', 'common') | generating random walks... | 20/50:   0%| | 97/264323 [58:35<2659:51:22, 3
100%|████████████████████████████████████████████████████████████████████▉| 7929600/7929690 [22:19:35<00:00, 98.66it/s]


In [34]:
walks = [w for w in walks if len(w) == len_random_walk]

In [35]:
out_file = open('random_walks.json', 'w')
json.dump(walks, out_file, indent=6)
out_file.close()


##### Word2vec

In [146]:
from gensim.models import Word2Vec

In [147]:
with open('random_walks.json', 'r') as file:
    node_contexts = json.load(file)
    

In [148]:
class Contexts():
    def __init__(self, data):
        self.data = data

    def __iter__(self):
        for context in tqdm(self.data, desc="Training..."):
            yield context

In [149]:
node_contexts = Contexts(node_contexts)
model = Word2Vec(node_contexts, vector_size=200, window=5, min_count=1, sg=1, workers=4, epochs=8)

Training...: 100%|████████████████████████████████████████████████████████| 7741140/7741140 [03:42<00:00, 34834.40it/s]


In [150]:
model.save("embeddings5.model")

#### Unweighted Graph

In [13]:
model1 = node2vec1.fit(window=10, min_count=1)

In [12]:
node2vec1 = Node2Vec(graph_original, dimensions=64, walk_length=30, num_walks=10, workers=4,  p=1, q=.25)

Computing transition probabilities: 100%|████████████████████████████████████| 264323/264323 [03:19<00:00, 1328.01it/s]


In [16]:
model1.save('unweighted_graph_embeddings.model')

In [39]:
wn.synsets('king')[0].definition()

'a male sovereign; ruler of a kingdom'

In [45]:
wn.synsets('easygoing')

[Synset('easy.s.02'), Synset('cushy.s.01'), Synset('easygoing.s.03')]

In [44]:
nltk.pos_tag(['easygoing'])

[('easygoing', 'VBG')]